# ✈️ Airline Passenger Satisfaction — Exploratory Data Analysis

**Project:** End-to-End Airline Passenger Satisfaction Analysis and Prediction  
**Dataset:** Kaggle Airline Passenger Satisfaction (103,904 train / 25,976 test rows)  
**Target:** `satisfaction` — `satisfied` (positive class) vs `neutral or dissatisfied`

---

> **Note on interpretation:** All findings in this notebook are *observational*. Correlations and group differences do not imply causation. Feature importance values describe the model's reliance on a feature, not the real-world causal effect on passenger satisfaction.


In [ ]:
import sys
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')
sys.path.insert(0, '..')

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.data_loader import build_dataset_summary, load_csv, validate_schema
from src.eda import *

plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
print('Libraries loaded successfully')


## 1. Data Loading & Schema Validation

In [ ]:
# Load datasets from project root
BASE = Path('..')
df_train = load_csv(BASE / 'dataset/train.csv')
df_test  = load_csv(BASE / 'dataset/test.csv')

# Validate schema
validate_schema(df_train, 'train')
validate_schema(df_test,  'test')

print(f'Train shape: {df_train.shape}')
print(f'Test shape:  {df_test.shape}')

In [ ]:
# Dataset overview
df_train.info()

In [ ]:
# First few rows
df_train.head()

In [ ]:
# Descriptive statistics
df_train.describe(include='all')

In [ ]:
# Missing values
missing = df_train.isnull().sum()
print('Missing values in training set:')
print(missing[missing > 0])

print('\nMissing values in test set:')
print(df_test.isnull().sum()[df_test.isnull().sum() > 0])

In [ ]:
# Duplicate check
print(f'Duplicate rows (train): {df_train.duplicated().sum()}')
print(f'Duplicate rows (test):  {df_test.duplicated().sum()}')

In [ ]:
# Unique value counts
print('Unique values per column (train):')
pd.DataFrame({'unique': df_train.nunique(), 'dtype': df_train.dtypes})

In [ ]:
# Machine-readable summary
summary = build_dataset_summary(df_train, df_test)
print('Target distribution (train):')
for k, v in summary['train']['target_distribution'].items():
    print(f'  {k}: {v["count"]:,} ({v["pct"]}%)')

## 2. Target Distribution

In [ ]:
fig = plot_target_distribution(df_train, output_dir=BASE / 'reports/figures')
plt.show()

**Interpretation:** The dataset is moderately imbalanced — ~56.7% neutral/dissatisfied vs ~43.3% satisfied. This imbalance is mild enough that standard models should handle it adequately, though it's worth monitoring precision/recall separately.

## 3. Passenger Demographics

In [ ]:
fig = plot_age_distribution(df_train, output_dir=BASE / 'reports/figures')
plt.show()

In [ ]:
fig = plot_satisfaction_by_gender(df_train, output_dir=BASE / 'reports/figures')
plt.show()

In [ ]:
fig = plot_satisfaction_by_customer_type(df_train, output_dir=BASE / 'reports/figures')
plt.show()

## 4. Travel Characteristics

In [ ]:
fig = plot_satisfaction_by_travel_type(df_train, output_dir=BASE / 'reports/figures')
plt.show()

In [ ]:
fig = plot_satisfaction_by_class(df_train, output_dir=BASE / 'reports/figures')
plt.show()

In [ ]:
fig = plot_class_travel_heatmap(df_train, output_dir=BASE / 'reports/figures')
plt.show()

In [ ]:
fig = plot_flight_distance_distribution(df_train, output_dir=BASE / 'reports/figures')
plt.show()

## 5. Service Ratings

In [ ]:
fig = plot_service_ratings_overview(df_train, output_dir=BASE / 'reports/figures')
plt.show()

In [ ]:
fig = plot_rating_distributions(df_train, output_dir=BASE / 'reports/figures')
plt.show()

## 6. Flight Delay Analysis

In [ ]:
print('Departure Delay statistics:')
print(df_train['Departure Delay in Minutes'].describe())
print(f'\n% of flights with zero delay: {(df_train["Departure Delay in Minutes"] == 0).mean()*100:.1f}%')
print(f'99th percentile: {df_train["Departure Delay in Minutes"].quantile(0.99):.0f} minutes')

In [ ]:
fig = plot_delay_distributions(df_train, output_dir=BASE / 'reports/figures')
plt.show()

In [ ]:
fig = plot_delay_scatter(df_train, output_dir=BASE / 'reports/figures')
plt.show()

## 7. Statistical Analysis

In [ ]:
fig = plot_correlation_heatmap(df_train, output_dir=BASE / 'reports/figures')
plt.show()

In [ ]:
fig = plot_satisfaction_rates_summary(df_train, output_dir=BASE / 'reports/figures')
plt.show()

## 8. Key EDA Findings Summary

| Finding | Detail |
|---------|--------|
| Class effect | Business class has the highest satisfaction rate; Eco has the lowest |
| Travel purpose | Business travelers are more satisfied than personal travel passengers |
| Customer loyalty | Loyal customers show higher satisfaction rates |
| Online boarding | Strongly associated with satisfaction — highest mean difference |
| Inflight entertainment | High ratings strongly associate with satisfaction |
| Seat comfort | Clear positive association with satisfaction |
| Delays | Long delays (>45 min) associate with lower satisfaction, but many on-time flights are still dissatisfied |
| Age | Middle-aged (31-60) passengers show higher satisfaction than young adults |
| Missing data | Only `Arrival Delay in Minutes` has 310 missing values (0.3%) — handled by median imputation |
| Duplicates | No duplicate rows found |

> **Caution:** All findings above are observational. We cannot claim that any feature *causes* satisfaction or dissatisfaction.